✅ Option 1: Pure asyncio Simulation (No Network)

In [ ]:

import asyncio
import time
import json
import random

# Simulate making a single async "query" with random latency and a JSON response
async def make_single_query():
    start_time = time.time()

    # Simulate network or compute latency
    simulated_latency = random.uniform(0.1, 0.8)  # seconds
    await asyncio.sleep(simulated_latency)

    # Simulate a structured JSON "response"
    # Randomly choose a vote to mimic model output
    simulated_response = {
        "vote": random.choice(["A", "B"]),
        "metadata": {
            "latency_ms": int(simulated_latency * 1000),
            "source": "simulated-async"
        }
    }

    # Parse response like you'd parse from a real API
    response_json = json.loads(json.dumps(simulated_response))
    vote = response_json.get('vote', '').strip()

    end_time = time.time()
    time_taken = end_time - start_time

    return vote, time_taken

async def run_multiple_queries_async(num_runs=10):
    start_time = time.time()

    tasks = [make_single_query() for _ in range(num_runs)]
    results = await asyncio.gather(*tasks)

    end_time = time.time()
    total_time = end_time - start_time

    # Tally and timing stats
    votes = {"A": 0, "B": 0}
    individual_times = []
    for vote, time_taken in results:
        if vote in votes:
            votes[vote] += 1
        individual_times.append(time_taken)

    avg_individual_time = sum(individual_times) / len(individual_times)

    print(f"\nResults after {num_runs} runs:")
    print(f"Total time: {total_time:.2f} seconds")
    print(f"Average time per run: {avg_individual_time:.2f} seconds")
    print(f"\nVote Tally:")
    print(f"Option A: {votes['A']} votes")
    print(f"Option B: {votes['B']} votes")

# Run the async function
if __name__ == "__main__":
    asyncio.run(run_multiple_queries_async(num_runs=10))


🌐 Option 2: aiohttp Against a Public Delay Endpoint

In [ ]:

import asyncio
import time
import json
import random
import aiohttp

# Hit a public delay endpoint and return a synthetic "vote"
async def make_single_query(session, delay_range=(0.1, 0.8)):
    start_time = time.time()

    # Pick a delay to request (server will hold the response for this long)
    delay = round(random.uniform(*delay_range), 2)

    # httpbin's /delay/{seconds} returns JSON after waiting that many seconds
    url = f"https://httpbin.org/delay/{delay}"

    async with session.get(url) as resp:
        # Status check (optional)
        if resp.status != 200:
            # Fall back to a default if something goes wrong
            content = {"error": f"HTTP {resp.status}"}
        else:
            content = await resp.json()

    # Simulate parsing to extract a "vote"
    # Random choice to mimic variability
    response_json = {
        "vote": random.choice(["A", "B"]),
        "httpbin": content
    }
    vote = response_json.get('vote', '').strip()

    end_time = time.time()
    time_taken = end_time - start_time

    return vote, time_taken

async def run_multiple_queries_async(num_runs=10):
    start_time = time.time()

    async with aiohttp.ClientSession(timeout=aiohttp.ClientTimeout(total=10)) as session:
        tasks = [make_single_query(session) for _ in range(num_runs)]
        results = await asyncio.gather(*tasks)

    end_time = time.time()
    total_time = end_time - start_time

    votes = {"A": 0, "B": 0}
    individual_times = []
    for vote, time_taken in results:
        if vote in votes:
            votes[vote] += 1
        individual_times.append(time_taken)

    avg_individual_time = sum(individual_times) / len(individual_times)

    print(f"\nResults after {num_runs} runs:")
    print(f"Total time: {total_time:.2f} seconds")
    print(f"Average time per run: {avg_individual_time:.2f} seconds")
    print(f"\nVote Tally:")
    print(f"Option A: {votes['A']} votes")
    print(f"Option B: {votes['B']} votes")

if __name__ == "__main__":
    asyncio.run(run_multiple_queries_async(num_runs=10))
